# Create a `dataset_id`-level split for cellxgene mouse data

The idea is to create a permanent split for cellxgene based on dataset_ids because validating on unseen datasets is a harder task.

Because the datasets are heterogeneous we want the training set to have seen all the celltypes. We also want the validation set to cover as many cell types as possible, without breaking the first rule.

One could imagine different splits that try and optimize the problem differently. This shows how to implement such logic for one case. The code can be modified to work with any csv file that has a split column.

Please note that as of March 2024, the TileDB-SOMA database backend does not work on CCC login nodes. In order to run this notebook, you must use a compute node. GPU is not necessary so a simple compute node like `jbsub -mem 32g -cores 8 -q x86_1h -interactive bash` would work.

If you only want to see the data, you can run this notebook locally (tested on MacOS with brew python) by setting `uri=None`.

## Loading dataset and exploration

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tiledbsoma

In [ ]:
census = tiledbsoma.open(
    uri="/dccstor/bmfm-targets/data/omics/transcriptome/scRNA/pretrain/cellxgene/soma-2023-12-15"
)

experiment = census["census_data"]["mus_musculus"]

To view the columns that are available for us to use for filtering the data using `value_filter` and/or to define a split, we'll read a single sample coalesce to pandas and print out the columns.  To learn more about using these operations, check out the notebook https://cloud.tiledb.com/notebooks/details/TileDB-Inc/a41fdf2c-6e69-415b-8598-1829b1998909/preview

In [ ]:
dataset_columns = (
    experiment.obs.read(
        coords=[0],
        value_filter="is_primary_data==True and nnz > 2",
    )
    .concat()
    .to_pandas()
    .columns
)
dataset_columns

The full documentation for these columns can be found in CellXGene's schema document
https://github.com/chanzuckerberg/single-cell-curation/blob/main/schema/3.0.0/schema.md

Let's see how many samples we have in this version of the dataset (`is_primary_data==True` removes duplicates)

In [ ]:
samples_df = (
    experiment.obs.read(
        value_filter="is_primary_data==True",
        column_names=[
            "dataset_id",
            "assay",
            "suspension_type",
            "cell_type",
            "donor_id",
            "tissue",
            "tissue_general",
            "nnz",
        ],
    )
    .concat()
    .to_pandas()
)

In [ ]:
samples_df.shape

In [ ]:
(samples_df.donor_id.value_counts() > 100)

It's worth noting that we have a variety of different assays here, and we may need to make use of this information at some point:

In [ ]:
samples_df[["assay", "dataset_id"]].drop_duplicates().groupby(
    "assay"
).count().squeeze().sort_values()

Not all datasets are the same size, not by a longshot. We actually have 6 orders of magnitude in dataset size. Yikes.

In [ ]:
samples_df.groupby("dataset_id").count().iloc[:, 0].hist(bins=np.logspace(0, 7, 20))
plt.xscale("log")

Let's look at the non-zero counts. How many samples have more than a certain number of non-zero reads? This is critically important for the purpose of context windows. Let's start by just plotting all the counts:

In [ ]:
nnz_counts = samples_df.nnz.value_counts()
nnz_counts.sort_index().plot()
plt.xscale("log")

In [ ]:
_ = plt.hist(samples_df.nnz, bins=1000, density=True, cumulative=True, histtype="step")
for h in [0.25, 0.5, 0.75]:
    plt.axhline(h, ls=":", color="orange")
plt.xscale("log")
plt.grid("on")
plt.xlabel("Number of nonzero reads")
plt.ylabel("Fraction of samples with fewer nonzero reads")
plt.tight_layout()
plt.xlim(xmax=samples_df.nnz.max())

In [ ]:
q = np.arange(0, 1.01, 0.05)
qvals = np.quantile(samples_df.nnz, q)
pd.Series(qvals, index=q, dtype=int)

From here we see that less than 15% of samples have only 512 nonzero reads! The good news is that getting from 1024 to 2048 covers a comfortable majority of all samples. Pushing above 4096 is definitely diminishing returns in terms of nonzeros.

## Create dataset x cell type dataframe

We want to be able to split the full CellxGene dataset by `dataset_id`, ie to use some component datasets for training and others for validation. Furthermore, since cell-types are particularly important we want to be sure that we have a good split of cell-types and not end up doing the entire training without encountering certain cell-types OR to run validation without checking certain cell-types. 

So we will create a dataframe that lists how many of every cell type we have for every dataset and use that for creating our split.

In [ ]:
def rowcolsort(df, ascending=False):
    row_sums = (df > 0).sum(axis=1)
    col_sums = (df > 0).sum(axis=0)
    return df.loc[
        row_sums.sort_values(ascending=ascending).index,
        col_sums.sort_values(ascending=ascending).index,
    ]


def make_dataset_celltype_df(samples_df, use_rowcolsort=True):
    dataset_df = samples_df.groupby(["dataset_id", "cell_type"]).count()
    dataset_df = dataset_df.iloc[:, 0].rename("sample_count").reset_index()
    pivoted_df = dataset_df.pivot_table(
        index=["dataset_id"], columns="cell_type", values="sample_count"
    ).reset_index()
    dataset_celltype_df = (
        pivoted_df.fillna(0)
        .applymap(lambda x: int(x) if isinstance(x, float) else x)
        .set_index("dataset_id")
    )
    if use_rowcolsort:
        dataset_celltype_df = rowcolsort(dataset_celltype_df, ascending=False)
    return dataset_celltype_df

In [ ]:
dataset_celltype_df = make_dataset_celltype_df(samples_df)

## Create celltype split

The logic behind this split is to try and maximize the samples in train while ensuring that every cell-type has at least one sample in validation. Since we are splitting by dataset, that means taking each dataset and assigning it to either validation or train based on the following check. If the dataset has cell-types which have never been seen before, assign it to train. If the dataset has cell-types which have been seen in train but not in validation, assign it to validation. If all of the dataset's cell-types have been seen in train and in validation, assign it to train.

The logic of the sorting is to start on the datasets with fewest celltypes and samples so that we can get validation taken care of without giving up too many samples from train.

One could definitely propose a more sophisticated process for assigning datasets which would optimize the twin objectives of maximal volume in training and maximal breadth in validation. But we didn't invest the time in that at this point. This split seems good enough and there's no such thing as a "perfect" split. The code base is open to future, superior splits.

In [ ]:
def add_cell_type_split_to_dataframe(dataset_celltype_df):
    train_cell_types = set()
    validation_cell_types = set()
    train_datasets = set()
    validation_datasets = set()
    for dataset_id, row in dataset_celltype_df.iterrows():
        cell_types = set(row[row > 0].index.to_list())
        if not cell_types.issubset(validation_cell_types) and cell_types.issubset(
            train_cell_types
        ):
            validation_cell_types.update(cell_types)
            validation_datasets.add(dataset_id)
        else:
            train_cell_types.update(cell_types)
            train_datasets.add(dataset_id)
    dataset_celltype_df.loc[list(train_datasets), "split"] = "train"
    dataset_celltype_df.loc[list(validation_datasets), "split"] = "dev"
    return dataset_celltype_df

In [ ]:
dataset_celltype_df = add_cell_type_split_to_dataframe(dataset_celltype_df)

Let's check how many samples we had in each split:

In [ ]:
split_counts = dataset_celltype_df.groupby("split").sum().sum(axis=1)
split_counts

In [ ]:
print(
    f"Fractions of total cells in splits:\n\t train: {split_counts['train'] / split_counts.sum()} \n\t dev: {split_counts['dev'] / split_counts.sum()}"
)

This is approximately what we would like, and it's doubtful we could do much better in terms of the split's effect on model quality. 

It is also instructive to look at the number of datasets in each split:

In [ ]:
dataset_celltype_df.split.value_counts()

Broadly speaking this is a good split. It avoids the problem of using samples that are too close to each other in train and validation, it guarantees full cell-type coverage in train, and it approximately maximizes cell-type coverage in validation. 

We will save it, and it will become our default split:

In [ ]:
dataset_celltype_df.to_csv("celltypes_split_mouse.csv")

## Deeper dive into split counts

If we want to dive in a little deeper, we can ask how many of every cell type there are in each split and in each dataset. Ideally, we would have more datasets and more samples of every cell-type in train than in dev. We see that that is almost, but not quite the case.

In [ ]:
split = dataset_celltype_df.pop("split")
split_dataset_count_df = (dataset_celltype_df > 0).groupby(split).sum().T
split_cell_count_df = dataset_celltype_df.groupby(split).sum().T
dataset_celltype_df = dataset_celltype_df.assign(split=split)
pd.concat([split_dataset_count_df, split_cell_count_df], axis=1)

In [ ]:
samples_df.suspension_type.value_counts()

In [ ]:
genes_df = (
    experiment.ms["RNA"].var.read(    )
    .concat()
    .to_pandas()
)

In [ ]:
genes_df

In [ ]:
!uv pip install gget

In [ ]:
import gget

orthdf = gget.bgee("ENSMUSG00000025902", type="orthologs")



In [ ]:
orthdf.query('genus == "Homo" & species == "sapiens"').drop_duplicates()

In [ ]:
gget.bgee?